# Notebook 3 — Jointures, agrégations, Parquet et PostgreSQL

In [ ]:
from collections import Counter
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("TradeCorp - Transformations")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

TMP_PATH = "/home/jovyan/data/tmp"
OUTPUT_PATH = "/home/jovyan/data/output"

def lire_parquet(nom):
    return spark.read.parquet(
        f"{TMP_PATH}/{nom}.parquet"
    )

df_customers = lire_parquet("customers")
df_orders = lire_parquet("orders")
df_order_details = lire_parquet("order_details")
df_products = lire_parquet("products")
df_categories = lire_parquet("categories")
df_suppliers = lire_parquet("suppliers")
df_employees = lire_parquet("employees")
df_shippers = lire_parquet("shippers")

print("DataFrames Parquet chargés avec succès.")

In [ ]:
dataframes = {
    "customers": df_customers,
    "orders": df_orders,
    "order_details": df_order_details,
    "products": df_products,
    "categories": df_categories,
    "suppliers": df_suppliers,
    "employees": df_employees,
    "shippers": df_shippers,
}

for nom, df in dataframes.items():
    print(nom, "=>", df.count(), "ligne(s)")

## Q21 Jointures orders et customers

In [ ]:
df_orders_customers = df_orders.alias("o").join(df_customers.alias("c"),F.col("o.customer_id")==F.col("c.customer_id"),"inner",).select("order_id","company_name","country","order_date","freight")

In [ ]:
df_orders_customers.show(10, truncate=False)

# Q22 Jointure order details et products 

In [ ]:
df_order_details_products = df_order_details.alias("od").join(df_products.alias("p"), F.col("od.product_id")==F.col("p.product_id"),"inner",).select("product_name","category_id","unit_price")

In [ ]:
df_order_details_products.show(10, truncate=False)

## Q23 Jointure products et categories 

In [ ]:
df_products_categories = (
    df_products.alias("p")
    .join(
        df_categories.alias("c"),
        F.col("p.category_id") == F.col("c.category_id"),
        "inner",
    )
    .select(
        F.col("p.*"),
        F.col("c.category_name"),
        F.col("c.description").alias(
            "category_description"
        ),
    )
)

df_products_categories.select(
    "product_id",
    "product_name",
    "category_name",
    "category_description",
).show(10, truncate=False)

## Q24A — Jointure complète sans renommage préalable

In [ ]:
df_products_categories_brut = (
    df_products
    .join(df_categories, "category_id", "inner")
)

df_jointure_brute = (
    df_order_details
    .join(df_orders, "order_id", "inner")
    .join(df_customers, "customer_id", "inner")
    .join(
        df_products_categories_brut,
        "product_id",
        "inner",
    )
    .join(df_employees, "employee_id", "left")
    .join(df_shippers, "shipper_id", "left")
)

compteur_colonnes = Counter(
    df_jointure_brute.columns
)

colonnes_dupliquees = {
    colonne: nombre
    for colonne, nombre in compteur_colonnes.items()
    if nombre > 1
}

print("Colonnes présentes plusieurs fois :")
print(colonnes_dupliquees)

## Q24B — Renommage des colonnes et reconstruction

In [ ]:
def renommer_colonnes(df, correspondances):
    resultat = df

    for ancien_nom, nouveau_nom in correspondances.items():
        if ancien_nom in resultat.columns:
            resultat = resultat.withColumnRenamed(
                ancien_nom,
                nouveau_nom
            )

    return resultat

In [ ]:
df_customers_renamed = renommer_colonnes(
    df_customers,
    {
        "company_name": "customer_company_name",
        "contact_name": "customer_contact_name",
        "contact_title": "customer_contact_title",
        "address": "customer_address",
        "city": "customer_city",
        "region": "customer_region",
        "postal_code": "customer_postal_code",
        "country": "customer_country",
        "phone": "customer_phone",
        "fax": "customer_fax",
    },
)

In [ ]:
df_products_renamed = renommer_colonnes(
    df_products,
    {
        "unit_price": "product_unit_price",
    },
)

df_categories_renamed = renommer_colonnes(
    df_categories,
    {
        "description": "category_description",
    },
)

df_products_enriched = (
    df_products_renamed
    .join(
        df_categories_renamed,
        "category_id",
        "inner",
    )
)

In [ ]:
df_employees_renamed = renommer_colonnes(
    df_employees,
    {
        "first_name": "employee_first_name",
        "last_name": "employee_last_name",
        "title": "employee_title",
        "hire_date": "employee_hire_date",
        "city": "employee_city",
        "country": "employee_country",
        "full_name": "employee_full_name",
    },
)

In [ ]:
df_shippers_renamed = renommer_colonnes(
    df_shippers,
    {
        "company_name": "shipper_name",
        "phone": "shipper_phone",
    },
)

In [ ]:
df_orders_enriched = (
    df_order_details
    .join(df_orders, "order_id", "inner")
    .join(
        df_customers_renamed,
        "customer_id",
        "inner",
    )
    .join(
        df_products_enriched,
        "product_id",
        "inner",
    )
    .join(
        df_employees_renamed,
        "employee_id",
        "left",
    )
    .join(
        df_shippers_renamed,
        "shipper_id",
        "left",
    )
)

df_orders_enriched.cache()

print(
    "Nombre de lignes enrichies :",
    df_orders_enriched.count()
)

print(
    "Nombre de colonnes :",
    len(df_orders_enriched.columns)
)

In [ ]:
compteur_final = Counter(
    df_orders_enriched.columns
)

doublons_finaux = {
    colonne: nombre
    for colonne, nombre in compteur_final.items()
    if nombre > 1
}

print("Colonnes dupliquées restantes :", doublons_finaux)

## Q25 — Chiffre d'affaires par client

In [ ]:
df_ca_client = (
    df_orders_enriched
    .groupBy("customer_company_name")
    .agg(
        F.round(
            F.sum("sous_total"),
            2
        ).alias("ca_total")
    )
    .orderBy(F.desc("ca_total"))
)

df_ca_client.show(10, truncate=False)

In [ ]:
#df_orders_enriched.show(10, truncate=False)

# Q26 CA par categorie 

In [ ]:
df_ca_categorie = (
    df_orders_enriched
    .groupBy("category_name")
    .agg(
        F.round(
            F.sum("sous_total"),
            2
        ).alias("ca_total"),
        F.countDistinct("product_id").alias(
            "nombre_produits_vendus"
        ),
    )
    .orderBy(F.desc("ca_total"))
)

df_ca_categorie.show(truncate=False)

# Q27 CA par mois 

In [ ]:
df_ca_mensuel = (
    df_orders_enriched
    .withColumn(
        "mois",
        F.date_trunc("month", F.col("order_date"))
    )
    .groupBy("mois")
    .agg(
        F.round(
            F.sum("sous_total"),
            2
        ).alias("ca_mensuel")
    )
    .orderBy("mois")
)

df_ca_mensuel.show(50, truncate=False)

## Q28 — Performance par employé

In [ ]:
df_ca_employe = (
    df_orders_enriched
    .groupBy("employee_full_name")
    .agg(
        F.round(
            F.sum("sous_total"),
            2
        ).alias("ca_total")
    )
)

In [ ]:
df_commandes_employe = (
    df_orders_enriched
    .select(
        "employee_full_name",
        "order_id",
        "order_date",
        "shipped_date",
    )
    .dropDuplicates(
        ["employee_full_name", "order_id"]
    )
    .withColumn(
        "delai_livraison_jours",
        F.datediff(
            F.col("shipped_date"),
            F.col("order_date")
        )
    )
    .groupBy("employee_full_name")
    .agg(
        F.countDistinct("order_id").alias(
            "nombre_commandes"
        ),
        F.round(
            F.avg("delai_livraison_jours"),
            2
        ).alias("delai_moyen_jours"),
    )
)

In [ ]:
df_performance_employes = (
    df_commandes_employe
    .join(
        df_ca_employe,
        "employee_full_name",
        "inner",
    )
    .orderBy(F.desc("ca_total"))
)

df_performance_employes.show(
    truncate=False
)

## Q29 — Window function : classement des produits

In [ ]:
df_ca_produit = (
    df_orders_enriched
    .groupBy(
        "category_name",
        "product_id",
        "product_name",
    )
    .agg(
        F.round(
            F.sum("sous_total"),
            2
        ).alias("ca_produit")
    )
)

fenetre_rang = (
    Window
    .partitionBy("category_name")
    .orderBy(F.desc("ca_produit"))
)

df_classement_produits = (
    df_ca_produit
    .withColumn(
        "rang_categorie",
        F.dense_rank().over(fenetre_rang)
    )
    .orderBy(
        "category_name",
        "rang_categorie"
    )
)

df_classement_produits.show(
    100,
    truncate=False
)

In [ ]:
df_classement_produits.filter(
    F.col("rang_categorie") <= 5
).show(100, truncate=False)

## Q30 — Chiffre d’affaires cumulé

In [ ]:
fenetre_cumul = (
    Window
    .orderBy("mois")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow,
    )
)

df_ca_cumule = (
    df_ca_mensuel
    .withColumn(
        "ca_cumule",
        F.round(
            F.sum("ca_mensuel").over(fenetre_cumul),
            2
        )
    )
)

df_ca_cumule.show(50, truncate=False)

## Q31 Tri et limites

In [ ]:
df_top_produits_quantite = (
    df_orders_enriched
    .groupBy(
        "product_id",
        "product_name",
    )
    .agg(
        F.sum("quantite").alias(
            "quantite_totale"
        )
    )
    .orderBy(F.desc("quantite_totale"))
    .limit(5)
)

df_top_produits_quantite.show(
    truncate=False
)

In [ ]:
df_top_pays = (
    df_orders_enriched
    .groupBy("customer_country")
    .agg(
        F.round(
            F.sum("sous_total"),
            2
        ).alias("ca_total")
    )
    .orderBy(F.desc("ca_total"))
    .limit(3)
)

df_top_pays.show(truncate=False)

## Q32 — Écriture du DataFrame enrichi en Parquet

In [ ]:
PARQUET_PATH = (
    f"{OUTPUT_PATH}/orders_enriched.parquet"
)

(
    df_orders_enriched
    .write
    .mode("overwrite")
    .parquet(PARQUET_PATH)
)

print(
    "Parquet créé :",
    PARQUET_PATH
)

# Notebook 4- Ecriture Parquet et chargement PosgreSQL bonus

## Q33 — Relecture du fichier Parquet

In [ ]:
df_parquet_verification = spark.read.parquet(
    PARQUET_PATH
)

nombre_original = df_orders_enriched.count()
nombre_parquet = df_parquet_verification.count()

print("Lignes du DataFrame original :", nombre_original)
print("Lignes du fichier Parquet :", nombre_parquet)
print("Nombres identiques :", nombre_original == nombre_parquet)

df_parquet_verification.printSchema()

## Q34- Comparer CSV vs Parquet

In [ ]:
CSV_COMPARISON_PATH = (
    f"{OUTPUT_PATH}/orders_enriched_csv"
)

(
    df_orders_enriched
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(CSV_COMPARISON_PATH)
)

In [ ]:
def taille_dossier(chemin):
    taille = 0

    for dossier, _, fichiers in os.walk(chemin):
        for fichier in fichiers:
            chemin_fichier = os.path.join(
                dossier,
                fichier
            )

            taille += os.path.getsize(
                chemin_fichier
            )

    return taille

taille_csv = taille_dossier(
    CSV_COMPARISON_PATH
)

taille_parquet = taille_dossier(
    PARQUET_PATH
)

print(
    "Taille CSV :",
    round(taille_csv / 1024, 2),
    "Ko"
)

print(
    "Taille Parquet :",
    round(taille_parquet / 1024, 2),
    "Ko"
)

if taille_csv > 0:
    gain = (
        1 - taille_parquet / taille_csv
    ) * 100

    print(
        "Gain de compression :",
        round(gain, 2),
        "%"
    )

### Comparaison CSV et Parquet

Parquet est généralement plus compact que CSV grâce à sa compression et à
son stockage en colonnes.

Il préserve également les types de données et permet à Spark de lire
uniquement les colonnes nécessaires. CSV stocke toutes les valeurs sous
forme de texte et nécessite une nouvelle inférence du schéma à chaque
lecture.

## Q35 — Partitionnement par pays

In [ ]:
PARTITION_PATH = (
    f"{OUTPUT_PATH}/orders_by_country"
)

df_partition_country = (
    df_orders_enriched
    .withColumn(
        "country",
        F.col("customer_country")
    )
)

(
    df_partition_country
    .write
    .mode("overwrite")
    .partitionBy("country")
    .parquet(PARTITION_PATH)
)

print("Parquet partitionné créé :", PARTITION_PATH)

In [ ]:
dossiers_pays = sorted(
    dossier
    for dossier in os.listdir(PARTITION_PATH)
    if dossier.startswith("country=")
)

print("Nombre de partitions :", len(dossiers_pays))

for dossier in dossiers_pays:
    print(dossier)

## Q36 Chargement PostgreSQL via JDBC

In [ ]:
print(
    "Packages Spark :",
    spark.sparkContext.getConf().get(
        "spark.jars.packages",
        "non défini"
    )
)

In [ ]:
jdbc_url = (
    "jdbc:postgresql://postgres:5432/tradecorp"
)

proprietes_postgres = {
    "user": "postgres",
    "password": os.getenv(
    "POSTGRES_PASSWORD",
    "postgres",
    ),
    "driver": "org.postgresql.Driver",
}

(
    df_orders_enriched
    .write
    .mode("overwrite")
    .jdbc(
        url=jdbc_url,
        table="orders_enriched",
        properties=proprietes_postgres,
    )
)

print(
    "Table PostgreSQL orders_enriched créée."
)

In [ ]:
import sys

if "/home/jovyan/src" not in sys.path:
    sys.path.append("/home/jovyan/src")

from reader import read_csv

df_test_reader = read_csv(
    spark,
    "/home/jovyan/data",
    "customers.csv",
)

print("Nombre de clients :", df_test_reader.count())
df_test_reader.show(5, truncate=False)